# Kiwi BM25 Retriever — 한국어 최적화 키워드 검색

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- 일반 BM25와 **KiwiBM25Retriever**의 차이, 특히 한국어 형태소 분석의 효과 이해하기
- `kiwipiepy`를 설치하고 `KiwiBM25Retriever`를 생성하는 방법 익히기
- 한국어 문서 검색에서 더 정확한 키워드 매칭이 왜 중요한지 파악하기

## 사전 지식

- BM25: TF-IDF를 개선한 키워드 빈도 기반 검색 알고리즘
- 형태소 분석(Morphological Analysis): 단어를 의미 있는 최소 단위로 분리

---

## 왜 Kiwi BM25인가?

### 일반 BM25의 한계 (한국어)

한국어 조사, 어미가 검색을 방해해요:

```
검색어: "머신러닝을 배워보자"
일반 BM25: "머신러닝을", "배워보자" → 정확히 같은 문자열만 매칭
KiwiBM25:  "머신러닝", "배우다" → 형태소로 분리 후 의미 중심 매칭
```

### Kiwi BM25의 장점

| 기능 | 설명 |
|------|------|
| 형태소 분석 | 명사, 동사, 형용사 등 추출 |
| 불용어 제거 | 조사, 어미 등 자동 제거 |
| 복합어 처리 | "머신러닝" → "머신"+"러닝" 분리 가능 |

```bash
pip install kiwipiepy
```

> **실무 팁**: KiwiBM25를 11번 노트북의 EnsembleRetriever와 결합하면 한국어 문서에서 더욱 강력한 하이브리드 검색이 가능해요.

In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
# ---------------------------------------------------
# Kiwi 형태소 분석 기반 BM25 Retriever 생성 (fallback: 일반 BM25)
# ---------------------------------------------------

# ============================================================
# TODO: KiwiBM25Retriever.from_documents()로 Retriever를 생성하고 한국어 쿼리를 검색하세요
# 힌트: kiwipiepy 미설치 시 BM25Retriever로 대체 (ImportError 처리)
# 예상 결과: 한국어 형태소 단위 검색 결과 출력
# ============================================================

try:
    from langchain_community.retrievers import KiwiBM25Retriever
    from langchain_community.document_loaders import TextLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    # 문서 로드
    loader = TextLoader("./data/ai-story.txt")
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    split_docs = text_splitter.split_documents(documents)
    
    # KiwiBM25Retriever: 한국어 형태소 분석 후 BM25 색인 생성
    # 일반 BM25보다 조사, 어미 처리가 정확하여 한국어 문서에 최적화

    
    # 검색
    query = "딥러닝에서 신경망 학습 방법은?"
    docs = kiwi_retriever.invoke(query)
    
    print(f"✅ Kiwi BM25 Retriever 생성 완료")
    print(f"\n📝 쿼리: {query}\n")
    print("="*80)
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        print(doc.page_content[:150] + "...")
        print("-"*80)
    
    print("\n💡 Kiwi BM25의 장점:")
    print("  - 한국어 형태소 분석으로 정확한 키워드 추출")
    print("  - 조사, 어미 등 불용어 자동 제거")
    print("  - 한국어 문서 검색에 최적화")
    
except ImportError:
    print("⚠️ kiwipiepy 패키지가 설치되지 않았습니다.")
    print("설치 명령: pip install kiwipiepy")
    print("\n대신 일반 BM25Retriever를 사용하겠습니다...")
    
    from langchain_community.retrievers import BM25Retriever
    from langchain_community.document_loaders import TextLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    loader = TextLoader("./data/ai-story.txt")
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    split_docs = text_splitter.split_documents(documents)
    
    # BM25Retriever: 공백 분리 기반 (한국어 조사/어미 처리 제한적)
    bm25_retriever = BM25Retriever.from_documents(split_docs)
    bm25_retriever.k = 3
    
    query = "딥러닝에서 신경망 학습 방법은?"
    docs = bm25_retriever.invoke(query)
    
    print(f"\n✅ 일반 BM25 Retriever 사용")
    print(f"\n📝 쿼리: {query}\n")
    print("="*80)
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        print(doc.page_content[:150] + "...")
        print("-"*80)

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | Kiwi 형태소 분석으로 한국어 토큰화 → BM25 키워드 검색 |
| 설치 | `pip install kiwipiepy langchain-community rank_bm25` |
| 일반 BM25와 차이 | 한국어 어절 단위가 아닌 형태소 단위로 색인 — 어미/조사 변형 처리 |
| 불용어 처리 | 분석 단계에서 품사 필터링으로 불필요한 조사·어미 제거 |
| 최적 사용 | 단독보다 FAISS와 EnsembleRetriever로 결합 시 효과 극대화 |

### 한국어 BM25 비교

| 방식 | 토큰화 기준 | 한국어 처리 | 권장 상황 |
|------|-------------|-------------|-----------|
| 일반 BM25 | 공백 분리 | 조사·어미 포함 색인 | 영어 중심 문서 |
| KiwiBM25 | 형태소 분리 | 의미 형태소만 색인 | 한국어 문서 |

### 다음 노트북 예고

**11-CC-EnsembleRetriever** — Contextual Compression(문맥 압축)과 EnsembleRetriever를 결합해 이 시리즈 최고 수준의 검색 파이프라인을 구성하는 방법을 배워요.